Test simplified model

In [ ]:
import numpy as np
import pandas as pd
import arviz as az
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec
from cmdstanpy import CmdStanModel
from sampler import CycloneDataSimulator, SimplifiedDataSimulator

In [ ]:
# test some prior parameters that I got from Claude:
a_sig = 3
b_sig = 0.2
a_tau = 3
b_tau = 0.1 
v2 = 0.25

# model parameters
B = 2               # number of ocean basins
D = 5               # number of predictors
N = 5000             # number of observations

# simulate N observations from the model
DataSimulator = SimplifiedDataSimulator(
    B = B,
    a_sig=a_sig, b_sig=b_sig,
    v2=v2,
    a_tau=a_tau, b_tau=b_tau,
    standardize=False
)
DataSimulator.generate_X(N_sim=N)
X = DataSimulator.X 
basins = DataSimulator.basins
W = DataSimulator.simulate()

# try standarxizing X before fitting
# X = (X - X.mean(axis=0)) / X.std(axis=0)


# save real parameters
real_params = [
    DataSimulator.sig2,
    DataSimulator.Beta,
    DataSimulator.tau2,
    DataSimulator.nu,
]

# load simulated data into a Stan dictionary
stan_basins = basins + 1            # Stan is 1-indexed!!!
stan_data = {
    "N" : N, "B" : B, "D" : D,
    "W" : W, "X" : X, "basins" : stan_basins,
    "a_sig" : a_sig, "b_sig" : b_sig,
    "v2" : v2,
    "a_tau" : a_tau, "b_tau" : b_tau,
}

In [ ]:
# compile the stan model
simple_model = CmdStanModel(stan_file="simplified_model.stan")

In [ ]:
# fit and sample using Stan model
fit = simple_model.sample(stan_data, iter_warmup=1000, iter_sampling=5000, chains=4)
draws_df = fit.draws_pd()

In [ ]:
param_names = [f"sig2[{b}]" for b in range(1, B+1)] + [f"Beta[{b},{d}]" for b in range(1, B+1) for d in range(1, D+1)] + [f"tau2[{d}]" for d in range(1, D+1)] + [f"nu[{d}]" for d in range(1, D+1)]
true_params = np.concatenate([np.atleast_1d(p).flatten() for p in real_params])

all_param_draws = []
param_stds = []
for param, true in zip(param_names, true_params):
    # normalizes the draws so that they can all be plotted together
    draws = draws_df[param].values[:]
    standardized_draws = (draws - true) / draws.std()
    all_param_draws.append(standardized_draws)
    param_stds.append(draws.std())

# I got help from Claude to make this plot because I didn't know how to plot all the parameters together in a nice way
fig = plt.figure(figsize=(14, 7))
fig.suptitle(f"Convergence of Posterior Parameter Draws, n={N} simulated observations",
             fontsize=13, fontweight='bold', y=1.01)

# Give boxplot ~75% of height, table ~25%
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.5)
ax = fig.add_subplot(gs[0])
ax_table = fig.add_subplot(gs[1])

# --- Boxplot ---
ax.boxplot(
    all_param_draws,
    tick_labels=param_names,
    whis=(5, 95),
    showfliers=False,
    # showmeans=True,
    # meanprops=dict(marker='x', markeredgecolor='black', markersize=6, markeredgewidth=1.5),
    medianprops=dict(color='steelblue', linewidth=1.5),
    boxprops=dict(color='steelblue'),
    whiskerprops=dict(color='steelblue'),
    capprops=dict(color='steelblue'),
)
ax.axhline(0, color='red', linestyle='--', linewidth=1, label='True value')
ax.set_ylabel("Std. Deviations from True Value", fontsize=10)
ax.tick_params(axis='x', rotation=90)
ax.legend(fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.set_xlim(0.5, len(param_names) + 0.5)

# --- Table ---
ax_table.axis('off')
table = ax_table.table(
    cellText=[[f'{s:.3f}' for s in param_stds]],
    colLabels=param_names,
    rowLabels=['Posterior STD'],
    loc='center',
    cellLoc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4)   # make rows a bit taller

# Style the header cells
for (row, col), cell in table.get_celld().items():
    if row == 0 or col == -1:
        cell.set_facecolor('#dce6f1')
        cell.set_text_props(fontweight='bold')
    cell.set_edgecolor('lightgray')
    
plt.show()

Test model with only one ocean basin

In [ ]:
# test some prior parameters that I got from Claude:
a_sig = 3
b_sig = 0.2
a_tau = 3
b_tau = 0.1 
v2 = 0.25

# model parameters
B = 1               # number of ocean basins
D = 5               # number of predictors
N = 500             # number of observations

# simulate N observations from the model
DataSimulator = SimplifiedDataSimulator(
    B = B,
    a_sig=a_sig, b_sig=b_sig,
    v2=v2,
    a_tau=a_tau, b_tau=b_tau,
    standardize=False
)
DataSimulator.generate_X(N_sim=N)
X = DataSimulator.X 
basins = DataSimulator.basins
W = DataSimulator.simulate()

# try standarxizing X before fitting
# X = (X - X.mean(axis=0)) / X.std(axis=0)


# save real parameters
real_params = [
    DataSimulator.sig2,
    DataSimulator.Beta,
    DataSimulator.tau2,
    DataSimulator.nu,
]

# load simulated data into a Stan dictionary
stan_basins = basins + 1            # Stan is 1-indexed!!!
stan_data = {
    "N" : N, "B" : B, "D" : D,
    "W" : W, "X" : X, "basins" : stan_basins,
    "a_sig" : a_sig, "b_sig" : b_sig,
    "v2" : v2,
    "a_tau" : a_tau, "b_tau" : b_tau,
}

# fit and sample using Stan model
fit = simple_model.sample(stan_data, iter_warmup=1000, iter_sampling=5000, chains=4)
draws_df = fit.draws_pd()

In [ ]:
param_names = [f"sig2[{b}]" for b in range(1, B+1)] + [f"Beta[{b},{d}]" for b in range(1, B+1) for d in range(1, D+1)] + [f"tau2[{d}]" for d in range(1, D+1)] + [f"nu[{d}]" for d in range(1, D+1)]
true_params = np.concatenate([np.atleast_1d(p).flatten() for p in real_params])

# I got help from Claude to make this plot because I didn't know how to plot all the parameters together in a nice way
fig = plt.figure(figsize=(14, 7))
fig.suptitle(f"Convergence of Posterior Parameter Draws, n={N} simulated observations",
             fontsize=13, fontweight='bold', y=1.01)

# Give boxplot ~75% of height, table ~25%
gs = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.5)
ax = fig.add_subplot(gs[0])
ax_table = fig.add_subplot(gs[1])

# --- Boxplot ---
ax.boxplot(
    all_param_draws,
    tick_labels=param_names,
    whis=(5, 95),
    showfliers=False,
    # showmeans=True,
    # meanprops=dict(marker='x', markeredgecolor='black', markersize=6, markeredgewidth=1.5),
    medianprops=dict(color='steelblue', linewidth=1.5),
    boxprops=dict(color='steelblue'),
    whiskerprops=dict(color='steelblue'),
    capprops=dict(color='steelblue'),
)
ax.axhline(0, color='red', linestyle='--', linewidth=1, label='True value')
ax.set_ylabel("Std. Deviations from True Value", fontsize=10)
ax.tick_params(axis='x', rotation=90)
ax.legend(fontsize=9)
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.set_xlim(0.5, len(param_names) + 0.5)

# --- Table ---
ax_table.axis('off')
table = ax_table.table(
    cellText=[[f'{s:.3f}' for s in param_stds]],
    colLabels=param_names,
    rowLabels=['Posterior STD'],
    loc='center',
    cellLoc='center',
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 1.4)   # make rows a bit taller

# Style the header cells
for (row, col), cell in table.get_celld().items():
    if row == 0 or col == -1:
        cell.set_facecolor('#dce6f1')
        cell.set_text_props(fontweight='bold')
    cell.set_edgecolor('lightgray')
    
plt.show()